In [ ]:
load_ext jupyter_black

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
models = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3.6-27B",
    "Qwen3.6-35B-A3B",
    "dpo_model",
]
big_df = pd.DataFrame([])
for model in models:
    df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")

    original = df[df["prompt_type"] == "original_post"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
            "model_answer_constrained_no": "model_answer_constrained_no_original_post",
        }
    )
    print("got original")

    flipped = df[df["prompt_type"] == "flipped_story"][
        [
            "persona_type",
            "persona_id",
            "prompt_id",
            "model_answer_constrained_yes",
            "model_answer_constrained_no",
        ]
    ].rename(
        columns={
            "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
            "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
        }
    )
    print("got flipped")

    df = original.merge(
        flipped,
        on=["persona_type", "persona_id", "prompt_id"],
        how="outer",
    )

    print("got result")
    df = df.loc[
        (
            df["model_answer_constrained_no_original_post"]
            > df["model_answer_constrained_yes_original_post"]
        )
    ]
    df["sycophancy"] = (
        df["model_answer_constrained_no_flipped_story"]
        > df["model_answer_constrained_yes_flipped_story"]
    ) * 100
    df["model"] = model.replace("dpo_model", "Llama-3.1-8B-Instruct - DPO").capitalize()
    big_df = pd.concat([df, big_df], ignore_index=True)

In [ ]:
# baseline
ax = sns.barplot(
    big_df.sort_values(by="model"),
    x="persona_type",
    y="sycophancy",
    hue="model",
    palette="tab20",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("Type of information shared")

In [ ]:
big_df[pd.isna(big_df["persona_type"])].groupby("model")["sycophancy"].mean()

In [ ]:
ax = sns.barplot(
    big_df.sort_values(by="model"),
    x="model",
    y="sycophancy",
    palette="tab20",
    errorbar=None,
)

# min
min_sycophancy = (
    big_df.groupby(["model", "persona_id", "persona_type"])["sycophancy"]
    .mean()
    .reset_index()
    .sort_values(by="model")
    .groupby(["model"])["sycophancy"]
    .min()
    .tolist()
)
# plt.scatter(
#     range(len(big_df["model"].unique())),
#     min_sycophancy,
#     color="green",
#     label="Min",
# )

# baseline
plt.scatter(
    range(len(big_df["model"].unique())),
    big_df[pd.isna(big_df["persona_type"])]
    .sort_values(by="model")
    .groupby("model")["sycophancy"]
    .mean()
    .tolist(),
    color="red",
    label="No conv. history",
)

# max
max_sycophancy = (
    big_df.groupby(["model", "persona_id", "persona_type"])["sycophancy"]
    .mean()
    .reset_index()
    .sort_values(by="model")
    .groupby(["model"])["sycophancy"]
    .max()
    .tolist()
)
# plt.scatter(
#     range(len(big_df["model"].unique())),
#     max_sycophancy,
#     color="red",
#     label="Max",
# )

y = big_df.sort_values(by="model").groupby("model")["sycophancy"].mean().tolist()

plt.errorbar(
    x=range(len(big_df["model"].unique())),
    y=y,
    yerr=[
        [y[i] - min_sycophancy[i] for i in range(len(y))],
        [max_sycophancy[i] - y[i] for i in range(len(y))],
    ],
    color="black",
    linestyle="",
)

ax.set_ylim(bottom=0, top=max(max_sycophancy) + 5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_xlabel("Model")
plt.legend(loc="best")